# Hardware Arithmetic - Verification by Hand
This notebook prints out a $3 \times 3$ grid of pixels, the $3 \times 3$ kernel weights, and shows the step-by-step arithmetic to calculate a single output pixel just like the hardware does.

In [ ]:
import numpy as np
from golden_conv import resolve_image, resolve_weights_dir, DATA_DIR
from generate_test_vectors import get_custom_filters
from PIL import Image
import json

image_path = resolve_image()
image = np.asarray(Image.open(image_path).convert("L"), dtype=np.uint8)

# Let's use the Sobel X filter (channel 1 in custom filters)
kernels, config = get_custom_filters()
kernel = kernels[1]
ch_config = config["channels"][1]
bias = int(ch_config["bias_quantized"])
shift = int(ch_config["shift"])
relu_en = bool(ch_config["relu_en"])

# Pick a 3x3 patch from the top-left of the image
patch = image[0:3, 0:3]
print("Pixel Patch (Unsigned 8-bit):")
print(patch)
print("\nKernel Weights (Signed 8-bit):")
print(kernel)

In [ ]:
accumulator = bias
print(f"Initial Accumulator (Bias) = {bias}\n")

for r in range(3):
    for c in range(3):
        p = int(patch[r, c])
        w = int(kernel[r, c])
        prod = p * w
        accumulator += prod
        print(f"Pixel({r},{c}): {p:3d} * {w:4d} = {prod:5d} | Acc = {accumulator}")

print(f"\nFinal Accumulator sum: {accumulator}")

In [ ]:
def shift_round_half_up(val: int, shift: int) -> int:
    if shift == 0: return val
    return (val + (1 << (shift - 1))) >> shift

shifted = shift_round_half_up(accumulator, shift)
print(f"Shift amount: {shift}")
print(f"After Right-Shift (Round-Half-Up): {shifted}")

# Saturate to 16-bit signed
saturated = max(-32768, min(32767, shifted))
print(f"After Saturation: {saturated}")

if relu_en:
    saturated = max(0, saturated)
    print(f"After ReLU: {saturated}")
else:
    print("ReLU disabled for this channel.")

print(f"\nFINAL OUTPUT PIXEL (Coordinate 0,0) = {saturated}")